In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [37]:
from pyspark.sql.functions import to_date, regexp_replace, col

path_csv = "/home/coder/data/"

df = spark.read.csv(path_csv, header=True, sep=',')

df_transform = (df
    .withColumnRenamed("tg_id", "telegram_id")
    .withColumn("user_nickname", regexp_replace(col("tg_nickname"), r"^@", ""))  # Убираем @
    .withColumn("registration_date", to_date(col("update_at")))  # TS → DATE
    .drop("pk_id","tg_nickname","update_at")
)

type_mapping = {
    "telegram_id": "long",
    "user_nickname": "string", 
    "registration_date": "date"
}

df_final = df_transform.select([
    col(c).cast(t).alias(c) for c, t in type_mapping.items()
])


# df_final = (df_transform
#     # telegram_id → UInt64 (LongType)
#     .withColumn("telegram_id", col("telegram_id").cast(LongType()))
#     # user_nickname → String
#     .withColumn("user_nickname", col("user_nickname").cast(StringType()))
#     # registration_date → Date
#     .withColumn("registration_date", col("registration_date").cast(DateType()))
# )

# if not df.isEmpty():
#     df_transformed.show()
#     df.printSchema() 


In [38]:
import clickhouse_connect

# ⬇️ Параметры подключения к CLICKHOUSE
jdbc_url = 'jdbc:clickhouse://clickhouse01:8123/avpalatov'
db_user = os.getenv('CLICKHOUSE_USER')
db_password = os.getenv('CLICKHOUSE_PASSWORD')
db_host = "clickhouse01"
cluster_name = "company_cluster"
ch_database = "avpalatov"
table_name = 'users_local'
distributed_table_name = 'users'

client = clickhouse_connect.get_client(host=db_host, port=8123, username=db_user, password=db_password)

drop_local =  f"""
    DROP TABLE IF EXISTS {ch_database}.{table_name} ON CLUSTER {cluster_name} SYNC
"""
create_local = f"""
    CREATE TABLE IF NOT EXISTS {ch_database}.{table_name} ON CLUSTER {cluster_name}
    (
        telegram_id         UInt64                 COMMENT 'id telegram',
        user_nickname       String                 COMMENT 'Никнейм',
        registration_date   Date                   COMMENT 'Дата регистрации'
    )
    ENGINE = ReplicatedMergeTree('/clickhouse/tables/{{shard}}/avpalatov_{table_name}', '{{replica}}')
    ORDER BY (telegram_id)
    COMMENT 'Регистрации кто хочет на буткемп'
    """

create_distributed = f"""
    CREATE TABLE IF NOT EXISTS {ch_database}.{distributed_table_name}
    AS {ch_database}.{table_name}
    ENGINE = Distributed('{cluster_name}', '{ch_database}', '{table_name}', telegram_id);
    """

# print(drop_local)
# print(create_local)
# print(create_distributed)

client.command(drop_local)
client.command(create_local)
client.command(create_distributed)

In [39]:
# ⬇️ Сохранение в CLICKHOUSE
df_final.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", distributed_table_name) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("truncate", "true") \
    .mode("append") \
    .save()

26/02/06 14:21:31 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/06 14:21:31 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/06 14:21:31 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/06 14:21:32 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/06 14:21:32 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/06 14:21:32 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [b3f9f09f-44ed-4c96-8655-4657019dca58](2 queries & 0 savepoints) is committed.
26/02/06 14:21:32 WARN Click

In [40]:
spark.stop()